# Long Context and Length Extrapolation


## 1. What is Extrapolation

**Extrapolation** = using patterns from a known range to predict what happens outside that range.

A real-life example:
- You have measured water temperature: 0°C → ice, 50°C → liquid, 100°C → boiling
- Now someone asks: what happens to water at 200°C? You haven't measured it, but based on the pattern you can infer → still gas
- That is extrapolation

LLMs face the same problem:
- During training: the model learned attention patterns for positions 0 to 4095
- During inference: the user provides a 10000-token article
- The question: can the model correctly handle tokens at positions 4096–9999, which it never saw during training?

**The answer depends on which position encoding you use.**

## 2. Position Encoding Review

(If you are already familiar with Embedding + Position from Part 3, you can skip this section. But RoPE later builds on this foundation, so if unsure it's worth a quick review.)

Attention itself is **order-agnostic**. Feed "cat sat mat" and "mat sat cat" to attention, and it computes the same attention scores — because attention only looks at "how related" tokens are, not "who comes before whom."

But order clearly matters. "I love you" and "you love I" mean completely different things.

**Position encoding attaches an "I am the Nth token" label to each token**, so that attention can use this information when computing relevance.

There are three ways to attach these labels:

In [ ]:
# Intuition: how order affects attention
print("Sentence A: I love you")
print("Sentence B: you love I")
print()
print("Without position encoding:")
print("  'I' and 'you' have the same attention score regardless of order")
print("  The model cannot distinguish 'I love you' from 'you love I'")
print()
print("With position encoding:")
print("  'I' at position 0 and position 2 gets different vectors -> attention can distinguish them")
print()
print("Here's the problem: training only saw up to 4096 positions,")
print("but inference receives 10000 positions -> what does the label for position 4097 look like?")

## 3. Extrapolation Capability of Three Position Encodings

| Method | How it works | Representative model | Can extrapolate? | Why? |
|:---|:---|:---|:---|:---|
| **Learned positions** | Randomly initialize a vector for each position during training, adjust during training | GPT-2 | ❌ No, not at all | Only learned vectors for positions 0–1023; the vector for position 1024 simply doesn't exist |
| **Sinusoidal encoding** | Use sin/cos functions to hand-compute each position's value, no learning needed | Original Transformer | Theoretically yes, practically poor | The functions are continuous, but the model hasn't learned to exploit that continuity |
| **RoPE (Rotary Position Encoding)** | Encode positions via "rotation"; position difference = rotation angle difference | LLaMA, Qwen, Mistral | ✅ **Yes!** | Relative positions are naturally extrapolatable, and frequency-domain properties can be exploited |

RoPE is now standard in nearly all open-source LLMs. Let's understand it below.

## 4. RoPE: Encoding Relative Position via Rotation

Section 2 reviewed the basic problem of position encoding: Attention itself doesn't distinguish "I love you" from "you love I". The sinusoidal position encoding solution adds a position vector on top of the Input Embedding, so each token's input representation carries position information.

RoPE (Rotary Position Embedding) takes a different approach. It does not modify the input Embedding, but directly intervenes in the dot product computation of Q and K — using rotation matrices to "write" relative position information into the dot product result.

### 4.1 Goal: make the Q and K dot product depend on relative position

Recall the core operation of Attention: the query vector q_m at position m and the key vector k_n at position n are dotted, and the dot product value determines the Attention weight between the two tokens.

Without position encoding, the dot product of q_m and k_n depends only on the semantic content of the two tokens. The same word pair, whether at the beginning or end of a sentence, gives the same dot product — because the vector looked up from Embedding depends only on the token ID, not on the position.

RoPE's goal is to design an operation f that injects position information into q and k, so that the dot product depends only on the **relative position m-n**:

$$f(q_m, m) \cdot f(k_n, n) = g(q_m, k_n, \boxed{m - n})$$

Why pursue "depends only on m-n"? Consider a concrete example. In the sentence "I love you", "love" is at position 1 and "you" is at position 2, a distance of 1. In the sentence "yesterday I love you", "love" moves to position 2 and "you" moves to position 3 — but the distance between "love" and "you" is still 1. In both sentences, the semantic relationship between "love" and "you" should be the same. If the dot product depends only on the relative distance, then word pairs at the same distance get the same Attention, and the model learns the most basic translational invariance in language.

Sinusoidal position encoding injects position via addition, but after addition the information is mixed together — q_m and k_n each contain the absolute positions m and n, and their dot product cannot cleanly depend only on m-n. RoPE achieves this clean dependency through rotation.

### 4.2 2D rotation: start from rotating a single vector pair

First look at the simplest case: the vector has only 2 dimensions. A point (x, y) on the plane rotated counterclockwise by angle θ is represented with the rotation matrix:

$$R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

The rotated result is:

$$\begin{bmatrix} x' \\ y' \end{bmatrix} = R_\theta \begin{bmatrix} x \\ y \end{bmatrix} = \begin{bmatrix} x\cos\theta - y\sin\theta \\ x\sin\theta + y\cos\theta \end{bmatrix}$$

Rotation does not change the vector's length: $\|R_\theta v\| = \|v\|$. This means rotation does not distort the token's semantic information — the vector's magnitude (representing semantic strength) stays unchanged; only its direction changes.

Now apply this to Attention. Rotate the query vector q_m at position m by an angle proportional to m, that is mθ, and rotate the key vector k_n at position n by nθ. Then compute the dot product:

$$(R_{m\theta}\, q_m)^T (R_{n\theta}\, k_n) = q_m^T R_{m\theta}^T R_{n\theta} k_n$$

The key step. Use two properties of rotation matrices:

**Property 1**: $R_\alpha^T = R_{-\alpha}$. The transpose of a rotation matrix equals reverse rotation — just negate the angle.

**Property 2**: $R_{-\alpha} R_\beta = R_{\beta-\alpha}$. First rotate backward by α°, then rotate forward by β°, is equivalent to directly rotating by (β-α)°.

Substituting gives:

$$q_m^T R_{-\alpha} R_\beta k_n = q_m^T R_{(n-m)\theta}\, k_n$$

On the right side only $n-m$ appears, not m or n individually. The absolute positions m and n cancel out, leaving only their difference.

**This means: after rotation, the dot product of q_m and k_n depends only on their relative position (n-m), not on where each is located.** Relative position information is naturally encoded into the Q and K dot product.

Below we manually verify this property with concrete numbers. Set two vectors q = (1, 1) and k = (1, 1), unit rotation angle θ = 15°, positions m=2, n=5 (distance 3). Also compute a second group m=10, n=13 (also distance 3) — if the two groups give the same dot product, "depends only on m-n" is verified.

In [ ]:
import math
import torch

# === Manual verification: the rotated dot product depends only on relative position ===
# Set q = (1, 1), k = (1, 1), θ = 15° = π/12
theta = math.pi / 12  # 15 degrees

def rotate_2d(v, angle):
    """Apply a rotation matrix to a 2D vector"""
    x, y = v[0], v[1]
    cos_a, sin_a = math.cos(angle), math.sin(angle)
    x_new = x * cos_a - y * sin_a
    y_new = x * sin_a + y * cos_a
    return torch.tensor([x_new, y_new])

q = torch.tensor([1.0, 1.0])
k = torch.tensor([1.0, 1.0])

# Case A: m=2, n=5 -> distance = 3
q_rot_A = rotate_2d(q, 2 * theta)
k_rot_A = rotate_2d(k, 5 * theta)
dot_A = torch.dot(q_rot_A, k_rot_A)

# Case B: m=10, n=13 -> distance also 3
q_rot_B = rotate_2d(q, 10 * theta)
k_rot_B = rotate_2d(k, 13 * theta)
dot_B = torch.dot(q_rot_B, k_rot_B)

# Case C: m=0, n=3 -> distance also 3 (more extreme contrast)
q_rot_C = rotate_2d(q, 0 * theta)
k_rot_C = rotate_2d(k, 3 * theta)
dot_C = torch.dot(q_rot_C, k_rot_C)

# Case D: m=2, n=6 -> distance = 4 (different distance, for comparison)
q_rot_D = rotate_2d(q, 2 * theta)
k_rot_D = rotate_2d(k, 6 * theta)
dot_D = torch.dot(q_rot_D, k_rot_D)

print("=== Manual verification: rotated dot product depends only on m-n ===")
print()
print("q = (1, 1), k = (1, 1), unit angle θ = 15°")
print()
print("Case A: m=2, n=5  (distance=3)    -> dot product =", f"{dot_A:.6f}")
print("Case B: m=10, n=13 (distance=3)   -> dot product =", f"{dot_B:.6f}")
print("Case C: m=0, n=3  (distance=3)    -> dot product =", f"{dot_C:.6f}")
print("Case D: m=2, n=6  (distance=4)    -> dot product =", f"{dot_D:.6f}")
print()
print("Key observations:")
print("  1. A, B, C have different m,n but all distance 3 -> identical dot product")
print("  2. D has distance 4 -> dot product differs from the first three groups")
print("  3. Verified: the dot product depends only on n-m, not on the absolute positions m and n")
print()
print("This is the core mathematical property of RoPE.")

# Bonus: change in dot product before and after rotation
dot_original = torch.dot(q, k)
print(f"\nBefore rotation: q·k = {dot_original:.4f}")
print(f"After rotation (distance=3): dot product ≈ {dot_A:.4f}")
print(f"The difference in relative position is encoded in the numeric change of the dot product")

## 5. Generalizing from 2D to d Dimensions

The previous section verified the principle of encoding relative position via rotation on 2D vectors. In practice, q and k usually have 64 or 128 dimensions. How do we generalize 2D rotation to d dimensions?

### 5.1 Grouped rotation of d-dimensional vectors

The scheme is more direct than it might seem: split d dimensions into d/2 non-overlapping 2D pairs:

```
Pair 0: (dim_0, dim_1)    -> rotates in its own 2D plane
Pair 1: (dim_2, dim_3)    -> rotates in its own 2D plane
...
Pair d/2-1: (dim_{d-2}, dim_{d-1}) -> rotates in its own 2D plane
```

Each pair rotates independently in its own 2D plane, and the rotations of different pairs don't interfere. Written in matrix form, the d-dimensional rotation matrix is a block-diagonal matrix — there are d/2 small 2×2 rotation matrices on the diagonal, and zeros everywhere else:

```
R = [R_{θ_0}    0      ...         0      ]
    [  0     R_{θ_1}   ...         0      ]
    [ ...    ...     ...         ...     ]
    [  0      0      ...  R_{θ_{d/2-1}}]
```

In actual code we don't really construct this large d×d matrix (most of it is zero, wasting compute and memory); instead we use vectorized operations: group adjacent dimensions in pairs, and compute each pair's rotation independently.

### 5.2 Each dimension pair has a different rotation speed

Using the same rotation speed for all dimension pairs doesn't work — all pairs would carry the same position information, wasting the expressive power of d dimensions. RoPE assigns different rotation speeds to different dimension pairs. The unit rotation angle of pair i is:

$$\theta_i = \frac{1}{10000^{2i/d}}, \quad i = 0, 1, ..., d/2-1$$

- **i=0** (first pair, dim_0 and dim_1): θ_0 = 1.0, rotates 1 radian (≈ 57°) per position, very fast. Within the 4096-position training window it completes 652 full rotations, so sin/cos have seen all possible values. Its job is to distinguish **adjacent** positions.
- **i=31** (last pair, dim_62 and dim_63, when d=64): θ_31 ≈ 0.00013, only about 0.008° per position, extremely slow. Within 4096 training positions it only covers about 0.55 radians (31°), far less than one rotation. Its job is to carry **long-range** position relationships.

This is the same high/low-frequency design idea as sinusoidal position encoding, but the mechanism is different: the sinusoidal scheme overlays different-frequency waveforms into one position vector added to the embedding; RoPE applies different-frequency rotations directly to Q and K, encoding relative distance through the dot product.

### 5.3 The full RoPE pipeline applied to Q and K

Now we can connect the full RoPE computation. Given a token sequence:

1. **Embedding + projection**: $x_0, x_1, ..., x_{L-1}$ go through Embedding and the Q/K/V projection matrices to give $q_m = W_Q x_m$, $k_n = W_K x_n$, $v = W_V x$

2. **Grouping**: each q and k is grouped into adjacent dimension pairs, (dim_0, dim_1) is the first pair, (dim_2, dim_3) is the second pair, and so on

3. **Pairwise rotation**: pair i of q at position m is rotated by angle $m \cdot \theta_i$, and pair i of k at position n is rotated by angle $n \cdot \theta_i$. Use the formulas from 4.2 to compute the rotated coordinates pair by pair

4. **Dot product**: the rotated q and k are dotted. According to the derivation in 4.2, pair i's contribution to the dot product contains $\cos((n-m)\theta_i)$, and the dot product of the whole vector is the sum of all pairs' contributions — the result depends only on the relative position $n-m$

5. **The rest is unchanged**: the softmax + multiply by V steps are exactly the same as in standard Attention

The structural difference from sinusoidal position encoding. Sinusoidal scheme: the position vector is added to the Embedding, and the subsequent Q/K projection mixes token semantics with position information. RoPE: position information bypasses the Embedding and acts directly on Q and K, precisely controlling the position dependency in the dot product via rotation matrices. V (Value) does not participate in rotation — because the Attention output doesn't need to carry position information through V; V only needs to provide the content to be "weighted and aggregated".

Below we implement the full RoPE in code and verify its relative-position property.

In [ ]:
# Direct look: how much do different dimension pairs' rotation speeds differ?
import torch
import math

d_k = 64          # 64 dimensions total, paired -> 32 pairs
base = 10000      # RoPE default base

pair_indices = torch.arange(0, d_k, 2).float()  # [0, 2, 4, ..., 62]
freqs = 1.0 / (base ** (pair_indices / d_k))

print(f"Total: {len(freqs)} dimension pairs")
print(f"Pair 0 (fastest) frequency: {freqs[0]:.4f}  -> rotates {math.degrees(freqs[0]):.1f}° per position")
print(f"Pair 16 (medium) frequency: {freqs[16]:.6f}  -> rotates {math.degrees(freqs[16]):.4f}° per position")
print(f"Pair 31 (slowest) frequency: {freqs[31]:.8f}  -> rotates {math.degrees(freqs[31]):.6f}° per position")

slowest_period = 2 * math.pi / freqs[31]
print(f"\nThe slowest pair needs {slowest_period:.0f} positions to complete one full rotation")
# -> Training window is 4096; the slowest hand hasn't even completed one rotation -> this is the extrapolation bottleneck


In [ ]:
# Visualize: different dimensions' hands moving with position (cos values)
import torch
import matplotlib.pyplot as plt

seq_len = 200
positions = torch.arange(seq_len).float()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax_idx, (pair_idx, label) in enumerate([
    (0, "Fast (second hand)"),
    (16, "Medium (minute hand)"),
    (31, "Slow (hour hand)")
]):
    theta = positions * freqs[pair_idx]
    ax = axes[ax_idx]
    ax.plot(positions.numpy(), theta.cos().numpy(), linewidth=1)
    ax.set_xlabel('Position'); ax.set_ylabel('cos(angle)')
    ax.set_title(f'Pair {pair_idx} — {label}\n{math.degrees(freqs[pair_idx]):.2f} deg per step')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Left: several full rotations (dense waveform) -> high frequency -> distinguishes neighbors
# Right: less than half a rotation (gentle curve) -> low frequency -> carries long-range information


## 6. Why Direct Extrapolation Fails

The longest sequence during training is 4096. Now at inference we get 8192 tokens. The simplest idea: let RoPE keep counting normally — position 4097, 4098, …, 8191, computing the rotation angle with m×θ_i as usual. Does it work?

The answer is: no. But understanding **why** it doesn't work is the key to grasping all extrapolation methods.

### 6.1 The difference between fast and slow dimensions

From Section 5 we know that different dimension pairs have vastly different rotation speeds. This means within the 4096 training positions, different dimensions cover different angle ranges and have different "experience". The table below lists several representative dimension pairs at d=64 within the training window:

| Dimension pair | Frequency θ_i | Total angle rotated in training window | Equivalent rotations | sin/cos range seen |
|:---|:---|:---|:---|:---|
| i=0 (fastest) | 1.0 | 4096 rad | ~652 rotations | All of [-1, 1], all shapes |
| i=8 | ~0.1 | ~410 rad | ~65 rotations | All of [-1, 1] |
| i=16 | ~0.01 | ~41 rad | ~6.5 rotations | All of [-1, 1] |
| i=24 | ~0.001 | ~4.1 rad | ~0.65 rotations | About 0.65 periods of sin/cos, ~65% of [-1, 1] |
| i=31 (slowest) | ~0.00013 | ~0.55 rad | ~0.09 rotations | About 0.09 periods of sin/cos, the [0, 0.52] interval |

**Core observation**: fast dimensions (i≤16) complete many full rotations within 4096 positions, and sin and cos have repeatedly seen every kind of value — rising, falling, peaks, troughs, inflection points — in [-1, 1]. They have sufficient training for any angle value.

Slow dimensions (i≥24) are different. The pair i=31 only rotates about 31° (0.55 rad) within the entire training window, less than 1/10 of one rotation. They have only seen the small interval where cos drops slowly from 1 to 0.85 and sin rises from 0 to 0.52. **Any angle value outside this interval is completely foreign to them.**

### 6.2 Extrapolation touches unfamiliar angles

When the inference length extends to 8192, position 8191's angle on the fastest dimension is 8191 radians, but since it has already completed many rotations, 8191 rad mod 2π still falls within the familiar range — the fast dimension has experience with any angle value.

The problem is with the slow dimensions. i=31's angle at position 8191 is 8191 × 0.00013 ≈ 1.1 rad (63°). Within the 4096 training window, this dimension has only seen up to 0.55 rad (31°). 63° far exceeds the training range — the values sin(63°) ≈ 0.89 and cos(63°) ≈ 0.45 have never been seen by the model on this dimension.

This leads to a key insight: **not all dimensions have extrapolation problems. Only the slow dimensions that didn't complete one rotation within the training window encounter unfamiliar angles at ultra-long positions.** The fast dimensions complete many rotations and have seen every angle, so they can extrapolate directly.

This explains why learned position encodings (GPT-2) can't extrapolate at all — each dimension is an independent value learned only within the training window, with no concept of "rotation"; all dimensions are equivalent to "slow dimensions that didn't complete one rotation". It also explains why sinusoidal position encoding can theoretically extrapolate but works poorly in practice — although all dimensions use sin/cos, the model struggles to learn to exploit this continuity-based extrapolation ability. RoPE, on the other hand, naturally exposes the frequency-domain structure of different dimensions, which happens to give us a lever we can operate on.

### 6.3 Understanding with a clock

Fast dimension: a second hand that has gone around 652 times — it has been everywhere, very experienced. Slow dimension: an hour hand that swings back and forth only between 0° and 31° — it only knows this small interval. Now ask it to point at 63°, and it can't make sense of it.

Below we plot the cosine value of i=31 (the slowest dimension) inside and outside the training window, to intuitively feel the "angle out of range" problem.

In [ ]:
# The problem with direct extrapolation: low-frequency dimensions exceed the trained angle range beyond the window
import torch
import matplotlib.pyplot as plt
import math

train_len, extrap_len = 4096, 8192
slow_pair = 31

positions_train = torch.arange(train_len).float()
positions_extrap = torch.arange(extrap_len).float()
theta_train = positions_train * freqs[slow_pair]
theta_extrap = positions_extrap * freqs[slow_pair]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(positions_extrap.numpy(), theta_extrap.cos().numpy(),
        linewidth=1, color='orange', label='cos value at inference')
ax.axvline(x=train_len, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.fill_between(range(train_len, extrap_len), -1.2, 1.2,
                alpha=0.1, color='red', label='unseen angle range')
ax.set_xlabel('Position'); ax.set_ylabel('cos(angle)')
ax.set_title(f'Direct extrapolation to 8192 (low-frequency dim #{slow_pair})\nRight of red line = unseen angle range')
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

# During training the angle was 0~{train_deg:.0f}°, after extrapolation it reaches {extrap_deg:.0f}°, exceeding by {over:.0f}°
train_deg = math.degrees(theta_train[-1].item())
extrap_deg = math.degrees(theta_extrap[-1].item())
print(f"Max angle during training: {train_deg:.1f}°  ->  Max angle after extrapolation: {extrap_deg:.1f}°  ->  Exceeded by {extrap_deg - train_deg:.1f}°")


## 7. Core Idea: Control the Angle Range

The conclusion of Section 6 is clear: extrapolation doesn't fail because RoPE itself is flawed, but because the slow dimensions' rotation angles exceed the training range. The fast dimensions have no problem.

**The solution: find a way to make the angles produced by positions 4096–8191 on the slow dimensions also fall within the [0, training max angle] interval seen during training. For the fast dimensions, intervene as little as possible — they have already seen every angle and need no extra processing.**

### 7.1 Intuition for three compression strategies

Imagine the model as a person who only recognizes house numbers 0 to 4095. Now we want it to recognize 4096 to 8191. There are three strategies:

**PI (Position Interpolation)**: divide all house numbers by 2. New 4096 maps back to old 2048, new 8191 maps back to old 4095 — all within the recognized range. The cost is that originally clearly distinct 1 and 2 now become 0.5 and 1, so the distinguishing ability drops.

**NTK-aware**: only compress the "less familiar" slow dimensions, and don't touch the "already familiar" fast dimensions. Through the adjustment of one parameter (the base value), it leverages the nonlinearity of the frequency formula to automatically complete the differentiated compression.

**YaRN**: further refines on top of NTK. Dimensions aren't a simple "fast" and "slow" binary split — the "neither fast nor slow" middle dimensions need a smooth transition. In addition, the softmax distribution of Attention changes after compression, requiring a temperature correction to calibrate.

### 7.2 The difference between the three methods from the frequency formula

All methods ultimately modify a parameter in this formula:

$$\theta_i = \frac{1}{10000^{2i/d}}$$

- **PI**: doesn't touch θ_i; instead replaces position m with m/scale. Equivalent to all θ_i shrinking by a factor of scale proportionally. All frequencies slow down together.

- **NTK-aware**: doesn't touch position m; instead replaces 10000 with $10000 \times scale^{d/(d-2)}$. Because the denominator is $base^{2i/d}$, the exponential dependency means: small i (fast dimensions) have small denominator changes -> frequency barely changes; large i (slow dimensions) have large denominator changes -> frequency drops a lot. One parameter achieves differentiation.

- **YaRN**: on top of NTK's base change, computes each dimension i's wavelength (the number of tokens needed to complete one rotation) $\lambda_i = 2\pi / \theta_i$. Based on the relationship between λ_i and the target length, it divides dimensions into three segments: dimensions with short λ_i aren't scaled, dimensions with long λ_i are scaled to scale×, and the middle dimensions transition smoothly. The temperature correction handles the change in Attention distribution sharpness.

The relationship among them isn't three parallel "methods", but a progressive path of deepening understanding: PI discovered "compression" -> NTK discovered "only compress the slow dimensions" -> YaRN discovered "compression needs segmented smoothing + temperature calibration".

The next three sections develop each method's specifics and code.

## 8. Method 1: Position Interpolation

**Paper**: Meta, 2023 — Extending Context Window via Position Interpolation

Idea: directly **proportionally compress** position indices.

```
Goal: extend the 4096 window to 8192
Scaling factor α = 4096 / 8192 = 0.5

New position = real position × 0.5

Real position 0    -> position given to model = 0 × 0.5 = 0
Real position 2048 -> position given to model = 2048 × 0.5 = 1024
Real position 8192 -> position given to model = 8192 × 0.5 = 4096 <- exactly at the training boundary!
```

**Analogy**: your street has house numbers 1 to 100, but you only recognize 1–50. Now numbers 51–100 arrive, and you divide all numbers by 2 — number 51 becomes 25.5, number 100 becomes 50, all within your recognized range.

**Cost**: all house numbers are compressed. Originally you could clearly distinguish number 1 from number 2; now 1 and 2 become 0.5 and 1 — the difference shrinks, so local resolution drops.

In [ ]:
# PI implementation: position × scaling factor, then compute RoPE normally
import torch
import matplotlib.pyplot as plt

train_len, target_len = 4096, 8192
alpha = train_len / target_len  # 0.5

pair_indices = torch.arange(0, 64, 2).float()
freqs_orig = 1.0 / (10000 ** (pair_indices / 64))
freqs_pi = freqs_orig * alpha

# Original RoPE vs PI compressed waveform
positions_orig = torch.arange(target_len).float()
angles_orig = positions_orig * freqs_orig[31]
positions_pi = torch.arange(target_len).float() * alpha
angles_pi = positions_pi * freqs_orig[31]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(angles_orig.cos().numpy(), linewidth=1, label='Original RoPE')
axes[0].plot(angles_pi.cos().numpy(), linewidth=1, label=f'PI (×{alpha:.2f})')
axes[0].axvline(x=train_len, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Position'); axes[0].set_ylabel('cos(angle)')
axes[0].set_title(f'Low-frequency dim #{31} wave\nPI stretches the wave (half frequency)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Right: all dimension frequencies are compressed proportionally
axes[1].plot(freqs_orig.numpy(), 'o-', markersize=3, label='Original frequency')
axes[1].plot(freqs_pi.numpy(), 's-', markersize=3, label='After PI scaling')
axes[1].set_xlabel('Dimension pair index (0=fast, 31=slow)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('PI scales all dimensions equally\nLocal resolution drops, light tuning needed')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Method 2: NTK-aware

**Paper**: NTK-Aware Scaled RoPE, bloc97, 2023

PI's flaw: it compresses **all** dimensions equally. But from Section 5, we know different dimensions rotate at different speeds:
- Fast dimensions (second hand): already completed many rotations within 4096 positions, so they've seen all kinds of angles -> **no compression needed**
- Slow dimensions (hour hand): haven't even completed one rotation within 4096 positions, many unseen angles -> **compression needed**

So NTK-aware's idea is: **only compress the slow ones, leave the fast ones alone.**

How? **Increase the base from 10000.** This is NTK-aware's most elegant insight:

```
Frequency formula: freq_i = 1 / base^(2i/d)

base = 10000 -> fast frequency -> slow dimension can't complete one rotation
base = 100000 -> frequency slows -> slow dimension rotates even slower -> smaller angles within the same positions -> doesn't exceed the training range!

Moreover:
  Low i (fast dimensions): freq ≈ 1 -> changing base barely affects them <- fast ones don't need adjustment
  High i (slow dimensions): freq ≈ 1/base -> changing base has a large effect <- slow ones get adjusted a lot
```

This precisely achieves "don't adjust fast hands, adjust slow hands a lot"! **Changing one parameter automatically accomplishes differentiated compression.**

In [ ]:
# Demonstrate NTK: effect of changing base on different dimensions
import torch
import matplotlib.pyplot as plt

base_old, scale = 10000, 2
# NTK formula: new base = old base × scale^(d/(d-2))
base_new = base_old * (scale ** (64 / 62))

pair_indices = torch.arange(0, 64, 2).float()
freqs_old = 1.0 / (base_old ** (pair_indices / 64))
freqs_new = 1.0 / (base_new ** (pair_indices / 64))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(freqs_old.numpy(), 'o-', markersize=3, label=f'base={base_old}')
axes[0].plot(freqs_new.numpy(), '^-', markersize=3, label=f'base={base_new:.0f}')
axes[0].set_xlabel('Dimension pair index (0=fast, 31=slow)'); axes[0].set_ylabel('Frequency')
axes[0].set_title('NTK-aware: increase base\nFast dims stay similar, slow dims slow down')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Compression ratio for each dimension
ratio = freqs_new / freqs_old
axes[1].bar(range(len(ratio)), ratio.numpy())
axes[1].set_xlabel('Dimension pair index'); axes[1].set_ylabel('New frequency / old frequency')
axes[1].set_title('Compression ratio by dimension\nFast dims ~100%, slow dims ~50%')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# NTK only needs to change base from 10000 to ~86000, no model structure change, and most cases need no fine-tuning


## 10. Method 3: YaRN

**Paper**: YaRN, 2023

NTK is already quite good, but YaRN identified one issue: after changing the base, the middle dimensions (neither fast nor slow) have attention that becomes "less decisive".

What does that mean? Recall the softmax step in attention:

```
softmax([2, 1, 0.5]) -> [0.59, 0.22, 0.13, 0.06]  <- relatively "sharp", attention concentrated
softmax([1, 0.5, 0.25]) -> [0.42, 0.26, 0.19, 0.14] <- relatively "flat", attention dispersed
```

Using "temperature" can adjust softmax sharpness:
```
Low temperature -> sharper softmax -> more concentrated attention -> good for local information
High temperature -> smoother softmax -> more dispersed attention -> good for long-range info (long-range doesn't need to be precise about which token anyway)
```

**YaRN's approach: NTK changes base + segmented scaling/adjustment for different dimension groups.** It is a common strong baseline, but not the "sole optimal" approach for all models and tasks. Later methods like LongRoPE and LongRoPE2 target even longer contexts.

- Fast dimensions: temperature = 1 (no adjustment, keep precision)
- Middle dimensions: smooth temperature transition
- Slow dimensions: slightly higher temperature (make long-range attention smoother)

In [ ]:
# YaRN's segmented strategy: divide dimensions into three groups by wavelength
import torch
import matplotlib.pyplot as plt
import math

scale, target_len = 4, 16384
pair_indices = torch.arange(0, 64, 2).float()
base_new = 10000 * (scale ** (64 / 62))
freqs_new = 1.0 / (base_new ** (pair_indices / 64))
wavelengths = 2 * math.pi / freqs_new  # positions needed for one full rotation

# Segmentation thresholds
low_bound = target_len / 1.0    # wavelength > this -> low frequency (needs scaling)
high_bound = target_len / 4.0   # wavelength < this -> high frequency (no scaling)

# ramping: smooth transition from 0 (no adjustment) to 1 (scale ×)
smooth = torch.clamp((wavelengths - high_bound) / (low_bound - high_bound), 0.0, 1.0)
dim_scale = (1 - smooth) * 1.0 + smooth * scale

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(32), dim_scale.numpy())
axes[0].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='No scaling')
axes[0].axhline(y=scale, color='red', linestyle='--', alpha=0.5, label=f'Scale {scale}x')
axes[0].set_xlabel('Dimension pair index (0=fast, 31=slow)'); axes[0].set_ylabel('Scale factor')
axes[0].set_title(f'YaRN: keep early pairs, scale later pairs {scale}x\nSmooth transition in the middle')
axes[0].legend()

axes[1].plot(wavelengths.numpy(), 'o-', markersize=3)
axes[1].axhline(y=high_bound, color='green', linestyle='--', alpha=0.5, label=f'High-frequency threshold ({high_bound:.0f})')
axes[1].axhline(y=low_bound, color='red', linestyle='--', alpha=0.5, label=f'Low-frequency threshold ({low_bound:.0f})')
axes[1].set_xlabel('Dimension pair index'); axes[1].set_ylabel('Wavelength (tokens per cycle)')
axes[1].set_yscale('log'); axes[1].set_title('Wavelength by dimension\nShort=high freq (unchanged), long=low freq (scaled)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# YaRN = NTK changes base + segmented smooth transition, a common strong baseline


## 11. Three Methods in One Sentence Each

| Method | One sentence | How it's done | Needs fine-tuning? |
|:---|:---|:---|:---|
| **PI** | Divide all house numbers by 2 | Position index × scaling factor | Yes |
| **NTK** | Slow down the clock's rotation speed | Increase RoPE's base value | No |
| **YaRN** | NTK + segmented processing for different dimensions | Change base + segmented smooth transition | Usually no or minimal fine-tuning |

**Most important practical knowledge**: in most cases, you just need to increase `rope_theta` (the base value) in the model config file:
- 4K -> 8K: change to around 500000
- 4K -> 32K: change to around 1000000

That's exactly what LLaMA 3 did — changed base from 10000 to 500000, going directly from 8K to 32K.

In [ ]:
# Complete code: a RoPE module supporting three extrapolation strategies
import torch
import torch.nn as nn

class ExtrapolatableRoPE(nn.Module):
    """RoPE supporting PI / NTK / YaRN extrapolation strategies"""

    def __init__(self, d_k, max_seq_len=4096, base=10000, strategy='ntk'):
        super().__init__()
        self.d_k = d_k
        self.max_seq_len = max_seq_len
        self.base = base
        self.strategy = strategy
        self._update_cache(max_seq_len, base)

    def _update_cache(self, seq_len, base, pi_scale=1.0):
        """Recompute the cos/sin cache"""
        positions = torch.arange(seq_len).float() / pi_scale
        freq = 1.0 / (base ** (torch.arange(0, self.d_k, 2).float() / self.d_k))
        angles = positions.unsqueeze(1) * freq.unsqueeze(0)
        cos = angles.cos().repeat_interleave(2, dim=-1)
        sin = angles.sin().repeat_interleave(2, dim=-1)
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)

    def set_extrapolation(self, target_len):
        """Set the extrapolation target length"""
        if target_len <= self.max_seq_len:
            return

        scale = target_len / self.max_seq_len

        if self.strategy == 'pi':
            self._update_cache(target_len, self.base, pi_scale=scale)
        elif self.strategy == 'ntk':
            new_base = self.base * (scale ** (self.d_k / (self.d_k - 2)))
            self._update_cache(target_len, new_base)
        elif self.strategy == 'yarn':
            new_base = self.base * (scale ** (self.d_k / (self.d_k - 2)))
            self._update_cache(target_len, new_base)  # Simplified; real YaRN also has temperature

        print(f"Extrapolation: {self.max_seq_len} -> {target_len} (strategy={self.strategy})")

    def forward(self, q, k, offset=0):
        """Apply rotation to Q and K"""
        seq_len = q.shape[-2]
        cos = self.cos[offset:offset + seq_len].to(q.device)
        sin = self.sin[offset:offset + seq_len].to(q.device)

        q_rot = q * cos + (torch.stack([-q[..., 1::2], q[..., ::2]], dim=-1).flatten(-2) * sin)
        k_rot = k * cos + (torch.stack([-k[..., 1::2], k[..., ::2]], dim=-1).flatten(-2) * sin)
        return q_rot, k_rot

# Test
rope = ExtrapolatableRoPE(d_k=64, max_seq_len=4096, strategy='ntk')
rope.set_extrapolation(32768)

q = torch.randn(1, 1, 100, 64)
k = torch.randn(1, 1, 100, 64)
q_rot, k_rot = rope(q, k)
print(f"Q: {q.shape} -> after rotation: {q_rot.shape}")

## 12. Methods for Validating Long Context

You've extended the context from 4K to 32K, but how do you prove it actually "understands" long text? You can't just go by feel.

**Probe test = design a question that can only be answered correctly by understanding the full text.** If the model answers correctly on a long text, it means its attention to distant tokens is effective.

#### 12.1 Needle in a Haystack — The Classic Test

The procedure is straightforward:
1. Generate a large body of irrelevant text (the "haystack"), say a 32K-token article
2. Insert a sentence at some position (the "needle"), like "the password is 12345"
3. Ask the model: "What is the password?"
4. If the model can find and correctly answer the password from the 32K text -> attention at that position is working well

Place the needle at different positions (beginning, middle, end), use different text lengths (1K, 2K, 4K, ..., 32K), test every combination -> draw a heatmap.

In [ ]:
# Needle in a Haystack test matrix (simulated): different lengths × different positions
context_lengths = [1024, 2048, 4096, 8192, 16384, 32768]
positions = [0.0, 0.25, 0.5, 0.75, 1.0]  # needle position (0=beginning, 1=end)

# Simulated results: ✅=correct, ❌=incorrect
results = [
    [True,  True,  True,  True,  True ],   # 1K
    [True,  True,  True,  True,  True ],   # 2K
    [True,  True,  True,  True,  True ],   # 4K
    [True,  True,  True,  True,  True ],   # 8K
    [True,  True,  False, True,  True ],   # 16K -- lost the middle
    [True,  False, False, True,  True ],   # 32K -- lost front-middle too
]

print("Needle in a Haystack test matrix:")
print(f"{'Length':<8}", *[f"{p:.0%} pos  " for p in positions])
for i, cl in enumerate(context_lengths):
    print(f"{cl:<8}", *[f"{'✅' if r else '❌'}   " for r in results[i]])
# Lost in the Middle: needles in the middle are more likely to be lost


#### 12.2 Tougher than Needle in a Haystack: RULER

Needle in a Haystack only tests the "find one sentence" ability. RULER is more comprehensive:

| Test | What it does | Why it's harder |
|:---|:---|:---|
| **Multi-needle recall** | Hide 3 different pieces of information, ask 3 times | Requires maintaining attention at multiple locations simultaneously |
| **Multi-hop reasoning** | Beginning says A=1, end says B=A+1, ask B | Requires combining two distant pieces of information to reason |
| **Variable tracking** | Track a value through multiple changes in the text | Requires updating memory |
| **Word frequency counting** | Count how many times a word appears in the full text | Requires traversing the full text and counting |

**Multi-hop reasoning is the most revealing**:
```
Text at 10% position: "Company A's revenue is 10 billion"
Text at 90% position: "Company B's revenue is 2× A's"
Question: "What is Company B's revenue?"

Model needs to:
  1. Find the information at 10% -> 10 billion
  2. Find the information at 90% -> 2×
  3. Combine and reason -> 20 billion
```
This is much harder than simply "finding one sentence", because it requires the model to maintain precise attention at both the beginning and the end simultaneously.

In [ ]:
# Multi-hop reasoning probe example: needs to remember multiple distant pieces of information simultaneously
print("=== Multi-hop reasoning probe (simulated) ===")
print(f"Text: 32K tokens")
print(f"  5%  position: Apples are 5 yuan each")
print(f"  50% position: Xiaoming buys 3")
print(f"  95% position: Spend 10+ get 2 off")
print(f"  Question: How much does Xiaoming pay? -> Answer: 5×3=15, spend 10+ get 2 off = 13 yuan")
print(f"\nRequires 5%->50%->95% three hops; missing any one gives the wrong answer")
# Answer 15 yuan -> missed the discount info at the end
# Answer "don't know" -> didn't find any information


#### 12.3 PPL Curves — Directly See "How Confused the Model Is"

**Perplexity (PPL)** = how confused the model is. Lower is better; lower means the model is more "confident" about the next token.

The best test: feed the same long text to the model and see whether PPL suddenly spikes at the training window boundary (4096).

In [ ]:
# Simulated PPL curves: how different extrapolation methods behave beyond the window boundary
import torch
import matplotlib.pyplot as plt

lengths = torch.linspace(512, 32768, 100)
ppl_none, ppl_pi, ppl_ntk = [], [], []

for L in lengths:
    if L <= 4096:
        ppl_none.append(10.0); ppl_pi.append(10.0); ppl_ntk.append(10.0)
    else:
        over = (L - 4096) / 4096
        ppl_none.append(10 + 100 * over**1.5)   # Explodes directly
        ppl_pi.append(10 + 8 * over**0.8)         # Rises slowly
        ppl_ntk.append(10 + 3 * over**0.5)        # Barely rises

plt.figure(figsize=(10, 5))
plt.plot(lengths.numpy(), ppl_none, label='No extrapolation', linewidth=2, color='red')
plt.plot(lengths.numpy(), ppl_pi, label='PI', linewidth=2, color='orange')
plt.plot(lengths.numpy(), ppl_ntk, label='NTK / YaRN', linewidth=2, color='green')
plt.axvline(x=4096, color='gray', linestyle='--', linewidth=1.5,
            alpha=0.7, label='Training window boundary (4K)')
plt.xlabel('Context length (tokens)'); plt.ylabel('PPL (lower is better)')
plt.title('PPL curves for extrapolation methods\nGood methods stay smooth past the boundary')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
# ✅ Good extrapolation: PPL stays smooth past 4096  |  ❌ Bad: PPL explodes violently


#### 12.4 Lost in the Middle — A Problem All Models Have

Even with extrapolation done well, one problem currently remains unsolved: **models naturally pay more attention to the beginning and end of text, while neglecting the middle.**

This is called the **"Lost in the Middle"** phenomenon.

Why? Because attention's softmax makes weights "compete" to sum to 1. The beginning and end have structural advantages:
- Beginning tokens are seen by all subsequent tokens (the starting point of causal attention)
- End tokens are closest to the current generation position (recency bias)
- Middle tokens are disadvantaged on both sides

In [ ]:
# Visualize "Lost in the Middle": middle information is naturally ignored
import torch
import matplotlib.pyplot as plt

seq_len = 8192; target_pos = 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ✅ Ideal: attention focuses on the key position
ideal_attn = torch.zeros(seq_len)
ideal_attn[target_pos] = 0.8
ideal_attn[max(0, target_pos-20):target_pos+20] += 0.01
ideal_attn /= ideal_attn.sum()
axes[0].plot(ideal_attn.numpy(), linewidth=0.5, color='green')
axes[0].axvline(x=target_pos, color='red', linestyle='--', alpha=0.7, label=f'Key position ({target_pos})')
axes[0].set_title('Ideal: attention focuses on target'); axes[0].legend(); axes[0].grid(True, alpha=0.2)

# ⚠ Reality: Lost in the Middle -> attention at edges, low in middle
u_shape = 1.0/(1+torch.arange(seq_len).float()) + 1.0/(1+torch.arange(seq_len-1,-1,-1).float())
u_shape /= u_shape.sum()
axes[1].plot(u_shape.numpy(), linewidth=0.5, color='purple')
axes[1].axvline(x=target_pos, color='red', linestyle='--', alpha=0.7, label=f'Key position ({target_pos})')
axes[1].set_title('Reality: Lost in the Middle'); axes[1].legend(); axes[1].grid(True, alpha=0.2)

# Recall rate: U-shaped curve
positions = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
recall_rates = [0.95, 0.85, 0.60, 0.40, 0.55, 0.85, 0.98]
axes[2].plot(positions, recall_rates, 'o-', markersize=8, linewidth=2, color='purple')
axes[2].set_xlabel('Information position'); axes[2].set_ylabel('Recall')
axes[2].set_title('Recall by information position\nU-shape: high at edges, low in middle'); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
# This is a structural problem, not something changing RoPE can fix. In practice, put important information at the edges.


## 13. Engineering Reality: Long Context is Not Just an Algorithm Problem

Even if position extrapolation methods work, long context has one hard bottleneck: **GPU memory and attention computation**.

Recall the KV Cache from Side Quest 14: for each new token generated, the K and V of all previous tokens must be stored. The longer the sequence, the larger the KV Cache:

```
4K context   -> KV Cache ≈ 2GB   (one consumer GPU is enough)
32K context  -> KV Cache ≈ 16GB  (A100 barely)
128K context -> KV Cache ≈ 64GB  (needs multi-GPU)
1M context   -> KV Cache ≈ 500GB (requires special techniques)
```

Engineering remedies:
- **KV Cache quantization**: store K and V in 8-bit or even 4-bit, saving 2–4× memory
- **Ring Attention**: split a long sequence into segments, distribute across multiple GPUs, each GPU handles one segment
- **StreamingLLM**: keep a few "anchor" tokens at the beginning + the most recent batch of tokens, discard the middle

### 13.1 Sliding Window Attention — Limiting Attention Range to Reduce Computation

PI, NTK, and YaRN mainly address the position encoding extrapolation problem: letting the model compute position representations beyond the training window; whether the quality is stable still depends on training and evaluation. But there's a more direct problem: even with perfect position encoding, the computation of standard causal attention itself makes long sequences impractical.

In standard causal attention, each token attends to all previous tokens. With sequence length N, the attention score matrix is N×N, and the computation grows as O(N²); the KV Cache itself is not O(N²) but grows linearly with sequence length: each layer stores K and V for every historical token. When N=128K, a single layer's attention matrix has 16B elements, exceeding 64GB in FP32 — this shows that both the intermediates and the computation of full attention are very expensive; meanwhile, the linearly growing KV Cache can also rapidly eat up memory for large, deep, highly concurrent models.

Sliding Window Attention takes a direct approach: each token no longer attends to all history, but only looks at the most recent W tokens. W is the window size, a fixed constant (e.g., 4096). This changes attention computation from O(N²) to O(W·N) — growing linearly with sequence length, not quadratically.

With W=4096 and N=128K, standard attention requires 16B dot products while Sliding Window needs only 524M — a 97% reduction. KV Cache can also shrink accordingly: each layer keeps K and V only for the most recent 4096 tokens instead of all 128K tokens, changing cache footprint from O(N) to O(W).

Implementation-wise, we just do one extra thing when constructing the attention mask: besides the causal mask (no future tokens), add a distance mask — if a key's position is more than W-1 away from the query, set it to -inf. Softmax naturally outputs 0 for -inf, which is equivalent to not attending.

The cost is losing long-range attention — token 100000 cannot see token 0 even if token 0 contains crucial information. Mistral 7B is a classic example: its docs and model notes emphasize sliding window attention with window size 4096. Other long-context models may mix in global attention, chunk attention, RingAttention, or retrieval-style memory; the exact window size and layer allocation must be checked in the corresponding model's config, not guessed from general rules.

Sliding Window and RoPE extrapolation solve two independent problems: RoPE helps the model "recognize" faraway positions, and Sliding Window makes the model "able to compute" sequences that long. The two are usually used together.


In [ ]:
import torch
import matplotlib.pyplot as plt

def create_sliding_window_mask(seq_len, window_size):
    """
    Construct a Sliding Window Attention mask.

    Each token i can only attend to tokens in the range [i - window_size + 1, i],
    while preserving the causal mask (cannot see the future).

    Returns: [seq_len, seq_len], 0 = allowed, -inf = blocked
    """
    # Standard causal mask (upper triangle = -inf)
    causal_mask = torch.triu(
        torch.full((seq_len, seq_len), float('-inf')), diagonal=1
    )

    # Find historical positions beyond the window
    row = torch.arange(seq_len).unsqueeze(1)  # [seq_len, 1]
    col = torch.arange(seq_len).unsqueeze(0)  # [1, seq_len]
    distance = row - col                      # [seq_len, seq_len]
    outside_window = distance > window_size - 1

    # Merge causal + sliding window
    mask = causal_mask.clone()
    mask[outside_window] = float('-inf')

    return mask

# Demo: masks with different window sizes
seq_len = 12
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax_idx, (w, title) in enumerate([
    (12, "Standard Causal Attention\n(W = N = 12)"),
    (6,  "Sliding Window\n(W = 6)"),
    (3,  "Sliding Window\n(W = 3)"),
]):
    mask = create_sliding_window_mask(seq_len, w)
    ax = axes[ax_idx]
    # Green = visible, red = invisible
    im = ax.imshow(mask, cmap='RdYlGn_r', aspect='auto', vmin=-10, vmax=0)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")

plt.tight_layout()
plt.show()

# Computation comparison
print("=== Sliding Window Computation Comparison ===")
print()
print(f"{'N':>8s}  {'Standard O(N²)':>14s}  {'SW O(W·N)':>14s}  {'Reduction':>8s}")
print("-" * 50)
for N in [4096, 8192, 32768, 131072]:
    W = 4096
    full_ops = N * N
    sw_ops = W * N
    reduction = (1 - sw_ops / full_ops) * 100
    print(f"{N:>8d}  {full_ops:>14,d}  {sw_ops:>14,d}  {reduction:>7.1f}%")

print()
print("Key observations:")
print("1. With a fixed window size W, computation grows linearly with N (O(W·N)), not quadratically")
print("2. When N >> W, the computation savings are very significant")
print("3. Sliding Window also saves KV Cache: each layer stores only W K,V pairs instead of N")
print("4. The cost is losing long-range attention -- occasional global attention layers are needed to compensate")

## 14. Hands-on: Extending 4K to 32K

```
Step 1: Choose a method
  -> Don't want to train -> NTK-aware (just change rope_theta)
  -> Willing to fine-tune -> YaRN (slightly better results)

Step 2: Change parameters
  -> Find rope_theta in the model's config.json
  -> 4K->32K reference value: change to 500000 ~ 1000000
  -> Or calculate by formula: new base = 10000 × (8)^(64/62) ≈ 86000

Step 3: Test
  -> Needle in a Haystack full-position heatmap
  -> RULER multi-hop reasoning
  -> PPL curve (should be smooth at the boundary)

Step 4: If not good enough
  -> Middle position recall is poor -> adjust YaRN's segmentation parameters
  -> Overall too high -> fine-tune on a small amount of long-text data
  -> Not enough GPU memory -> use KV Cache quantization + vLLM
```

## 15. Hands-on: ModelScope + NTK Extension

We've covered a lot of theory; now let's do it for real.

**Task**: Pull a Qwen model from ModelScope, check its default context configuration, calculate the `rope_theta` needed for extension using the NTK-aware method, then run a Needle in a Haystack test to verify whether the extension works.

In [ ]:
# === Optional hands-on dependency: transformers / modelscope ===
# The first half of this notebook is pure hand-written principle demos; this section
# automatically falls back to ToyModel if dependencies are missing.

import torch

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from modelscope import snapshot_download
    HAS_REAL_LONG_CONTEXT_DEMO = True
except ModuleNotFoundError as e:
    AutoModelForCausalLM = AutoTokenizer = snapshot_download = None
    HAS_REAL_LONG_CONTEXT_DEMO = False
    print(f"Optional dependency missing: {e}")
    print("Using ToyTokenizer/ToyModel to run through the rest; for real Qwen hands-on, install transformers + modelscope.")

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# === Download and load Qwen2.5-0.5B-Instruct from ModelScope ===
# If optional dependencies or network are unavailable locally, use ToyModel to keep the notebook runnable.

import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

if HAS_REAL_LONG_CONTEXT_DEMO:
    print("Downloading model from ModelScope...")
    model_dir = snapshot_download(model_name, revision="master")
    print(f"Model downloaded to: {model_dir}\n")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
        trust_remote_code=True,
    )
    if DEVICE == "cpu":
        model = model.to(DEVICE)
    model.eval()
else:
    print("Skipping real model download, creating offline ToyModel for data flow demo.")

    class ToyConfig:
        model_type = "toy-qwen"
        hidden_size = 1024
        num_hidden_layers = 2
        num_attention_heads = 16
        max_position_embeddings = 32768
        rope_theta = 1000000.0
        rope_scaling = None

    class ToyTokenizer:
        def __init__(self):
            self.eos_token_id = 0
            self.vocab = {"<eos>": 0}
            self.reverse = {0: ""}

        def encode(self, text, add_special_tokens=False):
            ids = []
            for ch in text:
                if ch not in self.vocab:
                    self.vocab[ch] = len(self.vocab)
                    self.reverse[self.vocab[ch]] = ch
                ids.append(self.vocab[ch])
            return ids

        def decode(self, ids, skip_special_tokens=False):
            return "".join(self.reverse.get(int(i), "") for i in ids)

        def apply_chat_template(self, messages, tokenize=False, add_generation_prompt=False):
            text = "".join(f"[{m['role']}] {m['content']}\n" for m in messages)
            if add_generation_prompt:
                text += "[assistant] "
            return self.encode(text, add_special_tokens=False) if tokenize else text

    class ToyModel:
        def __init__(self, tokenizer):
            self.config = ToyConfig()
            self.tokenizer = tokenizer

        def to(self, device):
            return self

        def eval(self):
            return self

        def generate(self, input_tensor, max_new_tokens=50, **kwargs):
            suffix = torch.tensor([self.tokenizer.encode("8842")], device=input_tensor.device)
            return torch.cat([input_tensor, suffix], dim=1)

    tokenizer = ToyTokenizer()
    model = ToyModel(tokenizer).to(DEVICE).eval()

print(f"Model type: {model.config.model_type}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Attention heads: {model.config.num_attention_heads}")
print(f"max_position_embeddings: {model.config.max_position_embeddings}  <- max position during training")
print(f"rope_theta: {model.config.rope_theta}  <- RoPE base value")
print(f"rope_scaling: {model.config.rope_scaling}  <- whether extrapolation strategy is enabled")

#### 15.1 Calculating rope_theta for NTK Extension

NTK formula: `new base = old base × scale^(d/(d-2))`

Where `scale = target length / original length`.

Below we calculate rope_theta for several common extension targets using Qwen2.5-0.5B:

In [ ]:
# === Use NTK formula to calculate rope_theta for different target lengths ===
# Qwen2.5-0.5B default: max_position = 32768, rope_theta = 1000000

def ntk_rope_theta(original_base, original_len, target_len, d_k):
    """NTK-aware: calculate rope_theta after extension"""
    scale = target_len / original_len
    new_base = original_base * (scale ** (d_k / (d_k - 2)))
    return new_base

# Qwen2.5-0.5B parameters
d_k = model.config.hidden_size // model.config.num_attention_heads  # head_dim
original_base = model.config.rope_theta
original_len = model.config.max_position_embeddings

print(f"Current config:")
print(f"  head_dim = {d_k}")
print(f"  rope_theta = {original_base:,}")
print(f"  max_position = {original_len:,} tokens")
print()

# Calculate rope_theta for extending to different lengths
targets = {
    "64K": 65536,
    "128K": 131072,
    "256K": 262144,
    "1M": 1048576,
}

print("NTK extension table:")
print(f"{'Target':<10} {'scale':<10} {'New rope_theta':<15} {'Formula'}")
print("-" * 65)
for label, target in targets.items():
    new_base = ntk_rope_theta(original_base, original_len, target, d_k)
    scale = target / original_len
    print(f"{label:<10} {scale:<10.1f} {new_base:<15,.0f} base×{scale:.1f}^({d_k}/{d_k-2})")

print(f"\nOperation: modify rope_theta in config.json to the corresponding value")
print(f"  Example: 4K->128K: change rope_theta from {original_base:,} to {ntk_rope_theta(original_base, original_len, 131072, d_k):,.0f}")

#### 15.2 Needle in a Haystack Test: Verifying Long Context Actually Works

Now construct a long text, hide a sentence in the middle, and see if the model can find it.

**Test design**:
1. Generate a "haystack" — fill with irrelevant text up to the target length
2. Insert a "needle" at a specified position — a key piece of information
3. Ask the model a question that can only be answered by reading the needle
4. See if the model can answer correctly

In [ ]:
# === Needle in a Haystack test ===

import torch

def build_needle_haystack(tokenizer, target_len, needle, needle_pos, question):
    """
    Construct a Needle in a Haystack test

    Args:
        target_len: target total token count
        needle: the information to hide (string)
        needle_pos: needle position (0~1, 0=beginning, 1=end)
        question: the question to ask
    """
    # Haystack: fill with a looping irrelevant text
    haystack_sentence = (
        "The quick brown fox jumps over the lazy dog. "
        "Machine learning is a subset of artificial intelligence. "
        "The weather today is quite pleasant with a gentle breeze blowing. "
        "Many people enjoy reading books and drinking coffee in the morning. "
    )

    # Encode haystack text to see how many tokens per sentence
    haystack_tokens = tokenizer.encode(haystack_sentence, add_special_tokens=False)
    repeat_times = (target_len // len(haystack_tokens)) + 2

    # Construct full text: haystack + needle insertion + haystack
    repeat_text = haystack_sentence * repeat_times
    full_tokens = tokenizer.encode(repeat_text, add_special_tokens=False)

    # Calculate insertion position (token level)
    insert_idx = int(target_len * needle_pos)
    needle_tokens = tokenizer.encode(f"\n\n[Important info]: {needle}\n\n", add_special_tokens=False)

    # Concatenate
    prefix = full_tokens[:insert_idx]
    suffix = full_tokens[insert_idx:target_len - len(needle_tokens)]
    test_tokens = prefix + needle_tokens + suffix
    test_tokens = test_tokens[:target_len]

    # Construct chat prompt
    test_text = tokenizer.decode(test_tokens)
    messages = [
        {"role": "system", "content": "You are a helpful assistant that extracts information. Answer briefly based on the text above."},
        {"role": "user", "content": f"Please read the following text and answer the question.\n\n{test_text}\n\nQuestion: {question}"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt, tokenizer.encode(prompt, add_special_tokens=False)

# Test parameters
target_len = 8000  # 8K token test text (within the default 32K window)
needle = "The combination to open the safe is 8842"
needle_pos = 0.5  # Place in the middle
question = "What is the combination to the safe?"

prompt, input_ids = build_needle_haystack(tokenizer, target_len, needle, needle_pos, question)
print(f"Test text length: {len(input_ids)} tokens")
print(f"Needle position: {needle_pos*100:.0f}% (approximately token {int(target_len*needle_pos)})")
print(f"Needle content: '{needle}'")
print(f"Question: '{question}'")
print()

# Generate answer
input_tensor = torch.tensor([input_ids]).to(DEVICE)
with torch.no_grad():
    output = model.generate(
        input_tensor,
        max_new_tokens=50,
        temperature=0.1,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

answer = tokenizer.decode(output[0][len(input_ids):], skip_special_tokens=True)
print(f"Model answer: {answer.strip()}")
print()

# Check if correct
if "8842" in answer or "8842" in answer.replace(" ", ""):
    print("✅ Test passed! Model successfully found the needle in 8K context")
else:
    print("❌ Test failed! Model did not correctly find the needle information")

#### 15.3 Full Position Scan: Needle in a Haystack Heatmap

Needle in a Haystack is the standard method for testing long-context capability. The procedure is to prepare a very long text (haystack), hide a fact (needle) at different depth positions — for example "the magic number is 78921" — then at the end ask "what is the magic number?". If the model answers correctly, it means it can attend to information at that depth.

A single test only gives one data point. A more thorough evaluation tests each position — e.g., at depths 0%, 25%, 50%, 75%, 100% — then plots the correctness at each position as a heatmap. The heatmap can intuitively show where the model is most likely to lose information: if the upper-left corner is blank and the lower-right is all green, the model isn't paying enough attention to information at the beginning; if the middle region is weak, there's a blind spot in attention. Below we draw this model's heatmap.

In [ ]:
# === Full-position Needle in a Haystack test ===

import torch

def test_needle_at_position(target_len, needle, needle_pos, question):
    """Run one needle test at a specified position, return whether successful"""
    prompt, input_ids = build_needle_haystack(tokenizer, target_len, needle, needle_pos, question)
    input_tensor = torch.tensor([input_ids]).to(DEVICE)

    with torch.no_grad():
        output = model.generate(
            input_tensor,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(output[0][len(input_ids):], skip_special_tokens=True)
    # Loose match: just need to produce the key number
    return "8842" in answer.replace(" ", "")

# Test at different positions within the default 32K window
positions = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
results = []

print(f"Needle in a Haystack test: text length={target_len} tokens")
print(f"{'Position':<12} {'Result':<8} {'Notes'}")
print("-" * 40)

for pos in positions:
    success = test_needle_at_position(target_len, needle, pos, question)
    results.append(success)
    desc = ""
    if pos < 0.2:
        desc = "(beginning -- easily visible)"
    elif pos > 0.8:
        desc = "(end -- recency bias)"
    else:
        desc = "(middle -- Lost in Middle high-risk zone)"
    print(f"{pos*100:3.0f}% pos    {'✅' if success else '❌'}      {desc}")

print()
success_rate = sum(results) / len(results)
print(f"Success rate: {success_rate:.0%} ({sum(results)}/{len(results)})")

# If middle positions fail, it confirms Lost in the Middle
mid_results = [r for p, r in zip(positions, results) if 0.15 < p < 0.85]
edge_results = [r for p, r in zip(positions, results) if p <= 0.15 or p >= 0.85]
if sum(mid_results) < len(mid_results):
    print(f"\n⚠ Middle position failures -> confirms Lost in the Middle phenomenon")
    print(f"   Edge success rate: {sum(edge_results)}/{len(edge_results)}")
    print(f"   Middle success rate: {sum(mid_results)}/{len(mid_results)}")

#### 15.4 Hands-on Summary

Through the demos above, we completed a full long-context extension verification workflow:

1. **Get model from ModelScope** -> `snapshot_download` + `AutoModelForCausalLM`
2. **Check default configuration** -> `rope_theta` and `max_position_embeddings` are the key parameters
3. **Calculate NTK extension** -> one formula `new_base = old_base × scale^(d/(d-2))`
4. **Modify configuration** -> change `rope_theta` in `config.json`; most cases don't require retraining
5. **Needle in a Haystack verification** -> full-position testing at target length, confirming the model can correctly recall information

**Production deployment steps**:
```bash
# 1. Modify config.json in the model directory
# Find "rope_theta": 1000000.0
# Change to the calculated new value, e.g., "rope_theta": 10000000.0

# 2. Use with inference framework
# vLLM: add --max-model-len 131072 at launch
# Transformers: directly load the modified config

# 3. Verify
# Run full-position Needle in a Haystack + PPL curve
```

**Key insight**: NTK's `rope_theta` change is just mathematical frequency compression — it doesn't change any model weights. It works because RoPE's frequency-domain structure naturally supports this kind of compression. This is RoPE's greatest advantage over learned position encodings (GPT-2) and sinusoidal encodings (original Transformer).

## Summary

1. ✅ **Extrapolation** = using patterns learned within the training window to handle positions beyond the window
2. ✅ **RoPE principle** = applies rotation matrices to Q and K so the dot product depends only on the relative position (m-n), not the absolute position
3. ✅ **2D rotation** -> d-dimensional generalization: a d-dimensional vector is split into d/2 pairs, each pair rotating independently in its own 2D plane
4. ✅ **Frequency difference** = different dimension pairs rotate at different speeds — fast dimensions complete many rotations (distinguishing neighbors), slow dimensions less than one rotation (carrying long-range relationships)
5. ✅ The essence of extrapolation failure: slow dimensions don't complete one rotation within the training window, so at inference the angle enters a region never covered during training
6. ✅ **PI** = proportionally compress all position indices (one-size-fits-all, sacrifices local resolution)
7. ✅ **NTK** = only change the base value, leveraging the nonlinearity of the frequency formula to automatically compress slow dimensions more and fast dimensions almost not at all
8. ✅ **YaRN** = NTK + segmented smooth transition by wavelength + temperature correction of the Attention distribution
9. ✅ The three methods form a progressive path: compress everything -> only compress the slow -> segmented smooth compression
10. ✅ In some scenarios you can try adjusting `rope_theta` first, but stable long context usually requires fine-tuning/continued training, evaluation, and inference framework cooperation
11. ✅ Probe tests: Needle in a Haystack, RULER (multi-needle multi-hop), PPL curves (check boundary smoothness)
12. ✅ **Lost in the Middle** = some models use information in the middle more weakly; placing important information at the edges is one mitigation strategy, but should be validated by task evaluation
13. ✅ Engineering also has the KV Cache memory bottleneck, requiring quantization/RingAttention/Sliding Window and other auxiliary methods

**One-sentence summary**: RoPE uses rotation matrices so the Q·K dot product naturally encodes relative position. The essence of extrapolation is to exploit RoPE's frequency-domain differences across dimensions — slow down the slow dimensions (so the angle doesn't exceed the training range), and keep the fast dimensions (to maintain local precision). NTK-aware achieves differentiated compression with one parameter via `rope_theta`; YaRN adds segmented smoothing and temperature calibration on top. Needle in a Haystack, RULER, and PPL curves are common stress tests, but cannot alone represent real long-context capability. References: [RoPE](https://arxiv.org/abs/2104.09864), [PI](https://arxiv.org/abs/2306.15595), [NTK-aware](https://www.reddit.com/r/LocalLLaMA/comments/14lz7j5/ntkaware_scaled_rope_allows_llama_models_to_have/), [YaRN](https://arxiv.org/abs/2309.00071), [LongRoPE](https://arxiv.org/abs/2402.13753), [LongRoPE2](https://arxiv.org/abs/2502.20082).

## Exercises

> You can ask AI to help explain the approach, but it's not recommended to let AI "do this problem for you" directly.

**Exercise 1: NTK-aware base value calculation**

The NTK-aware method implements extrapolation by modifying RoPE's base parameter, with the formula:

$$\text{new\_base} = \text{old\_base} \times \text{scale}^{d/(d-2)}$$

Suppose the original base = 10000, dimension $d = 128$ (head dimension), and you want to extend from 4K to 32K (scale = 8). Calculate the new base value.

Hint: $d/(d-2) = 128/126 \approx 1.016$, $\text{new\_base} = 10000 \times 8^{1.016}$.

In [ ]:
# Exercise 1: calculate the NTK-aware base
import math

old_base = 10000
d = 128
scale = 8

# TODO: calculate the exponent d/(d-2)
exponent = d / (d - 2)

# TODO: calculate the new base
new_base = old_base * scale ** exponent

assert exponent is not None, "Calculate the exponent first"
assert new_base is not None, "Calculate the new base first"

expected_exp = d / (d - 2)
expected_base = old_base * (scale ** expected_exp)
assert abs(exponent - expected_exp) < 0.001, f"The exponent should be {expected_exp:.4f}"
assert abs(new_base - expected_base) / expected_base < 0.01, f"The new base should be {expected_base:.0f}"

print("✅ Exercise 1 passed:")
print(f"   exponent d/(d-2) = {exponent:.4f}")
print(f"   new_base = {new_base:.0f}")
print(f"   this is {new_base/old_base:.1f}x the original base")
print("   By changing one parameter, NTK automatically compresses different dimensions by different amounts.")


**Exercise 2: RoPE rotation angle calculation**

RoPE applies a rotation to each dimension pair, with angle $\theta = m / \text{base}^{2i/d}$, where $m$ is the position index and $i$ is the dimension pair index. Suppose base = 10000 and $d = 4$ (2 dimension pairs). Calculate the rotation angle of the first dimension pair ($i=0$) at positions $m = 0$ and $m = 100$.

Hint: $\theta_{i=0} = m / \text{base}^{0} = m$ (radians), so at $m=100$, $\theta = 100$ radians.

In [ ]:
# Exercise 2: calculate RoPE rotation angles
import math

base = 10000
d = 4
i = 0

# TODO: calculate the rotation angle at position m=0
theta_m0 = 0 / (base ** (2 * i / d))

# TODO: calculate the rotation angle at position m=100
theta_m100 = 100 / (base ** (2 * i / d))

assert theta_m0 is not None, "Calculate the angle for m=0 first"
assert theta_m100 is not None, "Calculate the angle for m=100 first"

expected_0 = 0 / (base ** (2 * i / d))
expected_100 = 100 / (base ** (2 * i / d))
assert abs(theta_m0 - expected_0) < 0.001, f"The angle at m=0 should be {expected_0}"
assert abs(theta_m100 - expected_100) / max(expected_100, 0.001) < 0.01, f"The angle at m=100 should be {expected_100:.4f}"

print("✅ Exercise 2 passed:")
print(f"   m=0:   theta = {theta_m0:.4f} rad (the origin, with no rotation)")
print(f"   m=100: theta = {theta_m100:.4f} rad (about {theta_m100/(2*math.pi):.1f} turns)")
theta_slow = 100 / (base ** (2 * 1 / d))
print(f"   comparison, i=1 at m=100: theta = {theta_slow:.6f} rad (almost no rotation)")
print("   Fast dimensions (i=0) distinguish nearby positions; slow dimensions (i=1) carry distant relationships.")


**Exercise 3: PI vs NTK extrapolation strategy comparison**

Position Interpolation (PI) and NTK-aware are two different extrapolation strategies:
- **PI**: proportionally compresses position indices. When extending from 4K to 32K, all position indices are divided by 8.
- **NTK-aware**: only modifies the base parameter, leveraging the nonlinearity of the frequency formula to automatically achieve differentiated compression.

Analyze whether the following statement is correct:

> "PI applies the same compression ratio to all dimensions, so local position resolution drops; NTK-aware preserves local resolution by giving different dimensions different compression ratios."

Hint: PI directly compresses position indices (all dimensions treated equally); NTK changes the base so high frequencies (fast dimensions) are barely affected and low frequencies (slow dimensions) are compressed.

In [ ]:
# Exercise 3: compare PI and NTK extrapolation

# TODO: decide whether the statement is true; enter True or False
answer = True

assert answer is not None, "Enter True or False"
assert isinstance(answer, bool), "Enter True or False"

if answer:
    print("✅ Exercise 3 passed:")
    print("   PI divides every position by scale, so every dimension pair receives the same compression ratio.")
    print("   It shrinks nearby relative distances, such as between 0 and 1, by scale and reduces resolution.")
    print("   NTK-aware scaling changes the base, leaving fast, high-frequency dimensions nearly unchanged;")
    print("   it compresses slow, low-frequency dimensions more so their angles stay within the training range.")
else:
    print("The statement is true. Think again about how PI and NTK affect dimensions of different frequencies.")
